In [67]:
import pandas as pd 
from glob import glob 
import os 

In [36]:
def parse_performance_log(log_dir):
    with open(log_dir,'r') as f:
        lines = f.readlines()

    data_dict = {}
    for line in lines:
        line = line.strip()
        if not line:  # Skip empty lines
            continue
        if ': ' not in line:  # Skip lines without the expected format
            continue
        try:
            key, value = line.split(': ')
            data_dict[key] = float(value)
        except ValueError:
            # Skip lines that can't be parsed properly
            continue
    
    return data_dict

def parse_performance_logs(log_dirs):
    performance_data = {}

    for log_path in log_dirs:
        # Extract filename from path
        filename = os.path.basename(log_path)
        
        # Parse class name and split (train/test) from filename
        # Format: {class}_{split}_epoch_0_performance.txt
        parts = filename.replace('_epoch_0_performance.txt', '').split('_')
        class_name = '_'.join(parts[:-1])  # Handle classes with underscores
        split = parts[-1]  # train or test
        
        # Parse the performance log
        parsed_data = parse_performance_log(log_path)
        
        # Initialize nested dict structure if needed
        if class_name not in performance_data:
            performance_data[class_name] = {}
        
        # Store parsed data
        performance_data[class_name][split] = parsed_data
    
    return performance_data

# Parse performance logs

def create_performance_dataframe(performance_data):
    """Create a DataFrame from performance data with MultiIndex columns."""
    rows = []

    # Get all unique classes and metrics
    all_classes = list(performance_data.keys())
    all_metrics = set()
    for class_data in performance_data.values():
        for split_data in class_data.values():
            all_metrics.update(split_data.keys())
    all_metrics = sorted(list(all_metrics))

    # Create rows for each class
    for class_name in sorted(all_classes):
        row_data = {'class': class_name}
        
        # Add train metrics
        if 'train' in performance_data[class_name]:
            for metric in all_metrics:
                train_value = performance_data[class_name]['train'].get(metric, None)
                # Convert flops to giga units
                if metric == 'flops' and train_value is not None:
                    train_value = train_value / 1e9
                row_data[('train', metric)] = train_value
        else:
            for metric in all_metrics:
                row_data[('train', metric)] = None
        
        # Add test metrics
        if 'test' in performance_data[class_name]:
            for metric in all_metrics:
                test_value = performance_data[class_name]['test'].get(metric, None)
                # Convert flops to giga units
                if metric == 'flops' and test_value is not None:
                    test_value = test_value / 1e9
                row_data[('test', metric)] = test_value
        else:
            for metric in all_metrics:
                row_data[('test', metric)] = None
        
        rows.append(row_data)

    # Create DataFrame
    df = pd.DataFrame(rows)

    # Set class as index
    df = df.set_index('class')

    # Create MultiIndex columns
    train_cols = [('train', metric) for metric in all_metrics]
    test_cols = [('test', metric) for metric in all_metrics]
    multi_cols = train_cols + test_cols

    # Reorder columns to match the MultiIndex structure
    df = df[multi_cols]

    # Set proper MultiIndex for columns
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=['split', 'metric'])
    
    return df

In [ ]:
log_dir = '/Volume/VAD/LifeLongerAD_cu121/results/SPADE/MVTecAD/efficient_metrics-Continual_False-online_False/seed_2/performance_logs'
log_dirs = glob(os.path.join(log_dir,'*.txt'))
performance_data = parse_performance_logs(log_dirs)
df = create_performance_dataframe(performance_data)

# Display the table
df

In [47]:
pr_dir = '/Volume/VAD/LifeLongerAD_cu121/results/SPADE/MVTecAD/efficient_metrics-Continual_False-online_False/seed_2/performance_logs/performance_report.txt'

def parse_performance_report(lines):
    """Parse performance report lines into structured data"""
    data = {}
    current_class = None
    
    for line in lines:
        line = line.strip()
        
        # Skip empty lines and headers
        if not line or line.startswith('=') or line.startswith('Performance') or \
           line.startswith('Overall') or line.startswith('Total') or \
           line.startswith('Average') or line.startswith('-'):
            continue
        
        # Skip specific lines that don't contain useful data
        if 'Per-Class Statistics' in line or 'Max Training GPU Memory' in line or 'Max Inference GPU Memory' in line:
            continue
        
        # Check if this is a class name line (ends with colon)
        if line.endswith(':'):
            current_class = line[:-1]
            data[current_class] = {}
        elif current_class and ':' in line:
            # Parse metric lines
            key, value = line.split(':', 1)
            key = key.strip()
            value = value.strip()
            
            # Parse numeric values
            if 'samples/sec' in value:
                numeric_value = float(value.split()[0])
                data[current_class][key.lower().replace(' ', '_')] = numeric_value
            elif 'GB' in value:
                numeric_value = float(value.split()[0])
                data[current_class][key.lower().replace(' ', '_')] = numeric_value
            elif 'G' in value and 'flops' in key.lower():
                numeric_value = float(value.replace('G', ''))
                data[current_class][key.lower().replace(' ', '_')] = numeric_value
            elif 'M' in value and 'params' in key.lower():
                numeric_value = float(value.replace('M', ''))
                data[current_class][key.lower().replace(' ', '_')] = numeric_value
    
    return data

def create_efficiency_table(data):
    """Create efficiency metrics table from parsed data"""
    import pandas as pd
    
    # Define metrics mapping
    metric_mapping = {
        'training_throughput': 'Training Throughput (samples/sec)',
        'inference_throughput': 'Inference Throughput (samples/sec)', 
        'max_training_gpu_memory': 'Max Training GPU Memory (GB)',
        'max_inference_gpu_memory': 'Max Inference GPU Memory (GB)',
        'model_flops': 'Model FLOPs (G)',
        'model_params': 'Model Params (M)'
    }
    
    rows = []
    for class_name, metrics in data.items():
        row = {'Class': class_name}
        for metric_key, display_name in metric_mapping.items():
            row[display_name] = metrics.get(metric_key, None)
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df = df.set_index('Class')
    
    return df

# Parse the performance report
with open(pr_dir,'r') as f:
    lines = f.readlines()
performance_data = parse_performance_report(lines)
efficiency_df = create_efficiency_table(performance_data)

# Display the efficiency metrics table
efficiency_df


,Training Throughput (samples/sec),Inference Throughput (samples/sec),Max Training GPU Memory (GB),Max Inference GPU Memory (GB),Model FLOPs (G),Model Params (M)
Class,,,,,,
capsule,36.16,3.94,None,None,11.454,66.834
toothbrush,37.27,7.86,None,None,11.454,66.834
screw,69.04,3.12,None,None,11.454,66.834
grid,68.75,3.43,None,None,11.454,66.834
carpet,37.51,3.24,None,None,11.454,66.834
cable,36.24,4.05,None,None,11.454,66.834
zipper,73.76,3.57,None,None,11.454,66.834
wood,36.51,4.01,None,None,11.454,66.834
tile,49.45,3.75,None,None,11.454,66.834


In [51]:
data = {}
current_class = None
for line in lines:
    line = line.strip()
    
    # Skip empty lines and headers
    if not line or line.startswith('=') or line.startswith('Performance') or \
        line.startswith('Overall') or line.startswith('Total') or \
        line.startswith('Average') or line.startswith('-'):
        continue
    
    # Skip specific lines that don't contain useful data
    if 'Per-Class Statistics' in line or 'Max Training GPU Memory' in line or 'Max Inference GPU Memory' in line:
        continue
    
    # Check if this is a class name line (ends with colon)
    if line.endswith(':'):
        current_class = line[:-1]
        data[current_class] = {}
    elif current_class and ':' in line:
        # Parse metric lines
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        
        # Parse numeric values
        if 'samples/sec' in value:
            numeric_value = float(value.split()[0])
            data[current_class][key.lower().replace(' ', '_')] = numeric_value
        elif 'GB' in value:
            numeric_value = float(value.split()[0])
            data[current_class][key.lower().replace(' ', '_')] = numeric_value
            break 

In [ ]:
import pandas as pd
import re

def parse_performance_report(file_path):
    """
    Parse performance report file and extract overall and per-class statistics.
    
    Args:
        file_path (str): Path to the performance report file
        
    Returns:
        tuple: (overall_stats_dict, per_class_df)
    """
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Parse overall statistics
    overall_stats = {}
    overall_section = re.search(r'Overall Statistics:\s*-+\s*(.*?)\s*Per-Class Statistics:', content, re.DOTALL)
    if overall_section:
        overall_text = overall_section.group(1)
        for line in overall_text.strip().split('\n'):
            if ':' in line:
                key, value = line.split(':', 1)
                key = key.strip()
                value = value.strip()
                
                # Extract numeric values
                if 'samples/sec' in value:
                    overall_stats[key] = float(value.replace(' samples/sec', ''))
                elif 'GB' in value:
                    overall_stats[key] = float(value.replace(' GB', ''))
                else:
                    try:
                        overall_stats[key] = float(value)
                    except ValueError:
                        overall_stats[key] = value
    
    # Parse per-class statistics
    per_class_data = []
    
    # Find per-class section
    per_class_section = re.search(r'Per-Class Statistics:\s*-+\s*(.*)', content, re.DOTALL)
    if per_class_section:
        per_class_text = per_class_section.group(1)
        
        # Split by class names (lines that end with colon and have no leading spaces)
        class_blocks = re.split(r'\n(?=\w+:)', per_class_text.strip())
        
        for block in class_blocks:
            if not block.strip():
                continue
                
            lines = block.strip().split('\n')
            if not lines:
                continue
                
            # First line should be class name
            class_line = lines[0]
            if ':' not in class_line:
                continue
                
            class_name = class_line.replace(':', '').strip()
            
            # Initialize class data
            class_data = {'class': class_name}
            
            # Parse metrics for this class
            for line in lines[1:]:
                line = line.strip()
                if ':' in line:
                    key, value = line.split(':', 1)
                    key = key.strip()
                    value = value.strip()
                    
                    # Clean up metric names and extract values
                    if 'Training Throughput' in key:
                        class_data['training_throughput_samples_per_sec'] = float(value.replace(' samples/sec', ''))
                    elif 'Inference Throughput' in key:
                        class_data['inference_throughput_samples_per_sec'] = float(value.replace(' samples/sec', ''))
                    elif 'Max Training GPU Memory' in key:
                        class_data['max_training_gpu_memory_gb'] = float(value.replace(' GB', ''))
                    elif 'Max Inference GPU Memory' in key:
                        class_data['max_inference_gpu_memory_gb'] = float(value.replace(' GB', ''))
                    elif 'Model FLOPs' in key:
                        # Convert to numeric (remove G suffix)
                        flops_value = value.replace('G', '')
                        class_data['model_flops_g'] = float(flops_value)
                    elif 'Model Params' in key:
                        # Convert to numeric (remove M suffix and convert to actual number)
                        params_value = value.replace('M', '')
                        class_data['model_params_m'] = float(params_value)
            
            if len(class_data) > 1:  # More than just class name
                per_class_data.append(class_data)
    
    # Create DataFrame
    per_class_df = pd.DataFrame(per_class_data)
    if not per_class_df.empty:
        per_class_df = per_class_df.set_index('class')
    
    return overall_stats, per_class_df

# Example usage

file_path = '/Volume/VAD/LifeLongerAD_cu121/results/ProxyCore/MVTecAD/efficient_metrics-Continual_False-online_False/seed_2/performance_logs/performance_report.txt'

# Parse and display
overall_stats, per_class_df = parse_performance_report(file_path)

per_class_df

In [75]:
import pandas as pd
import re

def parse_performance_report(file_path):
    """
    Parse performance report file and extract overall and per-class statistics.
    
    Args:
        file_path (str): Path to the performance report file
        
    Returns:
        tuple: (overall_stats_dict, per_class_df)
    """
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Parse overall statistics (if present)
    overall_stats = {}
    overall_section = re.search(r'Overall Statistics:\s*-+\s*(.*?)\s*(?=\w+:)', content, re.DOTALL)
    if overall_section:
        overall_text = overall_section.group(1)
        for line in overall_text.strip().split('\n'):
            if ':' in line:
                key, value = line.split(':', 1)
                key = key.strip()
                value = value.strip()
                
                # Extract numeric values
                if 'samples/sec' in value:
                    overall_stats[key] = float(value.replace(' samples/sec', ''))
                elif 'GB' in value:
                    overall_stats[key] = float(value.replace(' GB', ''))
                else:
                    try:
                        overall_stats[key] = float(value)
                    except ValueError:
                        overall_stats[key] = value
    
    # Parse per-class statistics - looking for pattern: "class_name:"
    per_class_data = []
    
    # Split content by lines
    lines = content.split('\n')
    current_class = None
    current_data = {}
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check if this is a class name line (no leading spaces, ends with colon)
        if line.endswith(':') and not line.startswith(' ') and 'Statistics' not in line:
            # Save previous class data if exists
            if current_class and current_data:
                current_data['class'] = current_class
                per_class_data.append(current_data)
            
            # Start new class
            current_class = line[:-1]  # Remove colon
            current_data = {}
            
        elif ':' in line and current_class:
            # This is a metric line for the current class
            key, value = line.split(':', 1)
            key = key.strip()
            value = value.strip()
            
            # Clean up metric names and extract values
            if 'Training Throughput' in key:
                current_data['training_throughput_samples_per_sec'] = float(value.replace(' samples/sec', ''))
            elif 'Inference Throughput' in key:
                current_data['inference_throughput_samples_per_sec'] = float(value.replace(' samples/sec', ''))
            elif 'Max Training GPU Memory' in key:
                current_data['max_training_gpu_memory_gb'] = float(value.replace(' GB', ''))
            elif 'Max Inference GPU Memory' in key:
                current_data['max_inference_gpu_memory_gb'] = float(value.replace(' GB', ''))
            elif 'Model FLOPs' in key:
                # Handle formats like "16.810M"
                if 'M' in value:
                    flops_value = value.replace('M', '')
                    current_data['model_flops_m'] = float(flops_value)
                elif 'G' in value:
                    flops_value = value.replace('G', '')
                    current_data['model_flops_g'] = float(flops_value)
            elif 'Model Params' in key:
                # Handle formats like "16.804M"
                if 'M' in value:
                    params_value = value.replace('M', '')
                    current_data['model_params_m'] = float(params_value)
                elif 'G' in value:
                    params_value = value.replace('G', '')
                    current_data['model_params_g'] = float(params_value)
    
    # Don't forget the last class
    if current_class and current_data:
        current_data['class'] = current_class
        per_class_data.append(current_data)
    
    # Create DataFrame
    per_class_df = pd.DataFrame(per_class_data)
    if not per_class_df.empty:
        per_class_df = per_class_df.set_index('class')
    
    return overall_stats, per_class_df

# Example usage
method = 'SPADE'
file_path = f'/Volume/VAD/LifeLongerAD_cu121/results/{method}/MVTecAD/efficient_metrics-Continual_False-online_False/seed_2/performance_logs/performance_report.txt'

# Parse and display
overall_stats, per_class_df = parse_performance_report(file_path)

per_class_df

,training_throughput_samples_per_sec,inference_throughput_samples_per_sec,max_training_gpu_memory_gb,max_inference_gpu_memory_gb,model_flops_g,model_params_m
class,,,,,,
capsule,36.16,3.94,0.284,0.258,11.454,66.834
toothbrush,37.27,7.86,0.537,0.509,11.454,66.834
screw,69.04,3.12,0.789,0.760,11.454,66.834
grid,68.75,3.43,1.027,1.011,11.454,66.834
carpet,37.51,3.24,1.288,1.263,11.454,66.834
cable,36.24,4.05,1.542,1.513,11.454,66.834
zipper,73.76,3.57,1.784,1.763,11.454,66.834
wood,36.51,4.01,2.040,2.015,11.454,66.834
tile,49.45,3.75,2.280,2.265,11.454,66.834
